# ============================================================
# Energy Use Survey – Create "Energy_Params" Sheet
# ============================================================

In [ ]:
# ----- STEP 1: Mount Google Drive -----
from google.colab import drive
drive.mount('/content/drive')

import openpyxl
import re
import datetime

Mounted at /content/drive


In [ ]:
# ----- STEP 2: Set file path -----
# Update this to wherever your file lives in Drive
FILE_PATH = '/content/drive/My Drive/MTP/Dataset/DIP_Data.xlsx'

wb = openpyxl.load_workbook(FILE_PATH)
ws = wb['Energy Use'] #change is different for other data files
MAX_COL = ws.max_column
print(f"Loaded sheet | columns: {MAX_COL}")

Loaded sheet | columns: 372


# ============================================================
# UTILITIES
# ============================================================

In [ ]:
def col_letters_to_num(letters):
    """Convert Excel column letters ('A','AB','AAA') to 1-based column number."""
    num = 0
    for ch in letters.upper():
        num = num * 26 + (ord(ch) - ord('A') + 1)
    return num


def parse_cell_ref(ref):
    """Parse 'AB12' → (row=12, col=28). Returns None if not a cell ref."""
    m = re.match(r'^([A-Za-z]+)(\d+)$', ref.strip())
    if m:
        return int(m.group(2)), col_letters_to_num(m.group(1))
    return None


def get_numeric(ws, row, col, visited=None):
    """
    Return the numeric value of a cell.
    - Evaluates Excel formulas recursively (handles cross-cell references).
    - datetime values (Excel mis-parsed ranges like '3-4') → (day+month)/2.
    - NA / NIL / None → 0.
    """
    if visited is None:
        visited = set()
    key = (row, col)
    if key in visited:
        return 0.0

    val = ws.cell(row, col).value

    if val is None:
        return 0.0
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, datetime.datetime):
        # Excel parsed a range like "3-4" as a date → recover original range average
        return (val.day + val.month) / 2.0
    if not isinstance(val, str):
        return 0.0

    val = val.strip()
    if val.upper() in ('NA', 'N/A', 'NIL', 'NULL', ''):
        return 0.0

    # Evaluate formula
    if val.startswith('='):
        visited2 = visited | {key}
        expr = val[1:]

        def replace_ref(m):
            parsed = parse_cell_ref(m.group(0))
            if parsed is None:
                return '0'
            r, c = parsed
            return str(get_numeric(ws, r, c, visited2))

        expr_resolved = re.sub(r'[A-Za-z]+\d+', replace_ref, expr)
        try:
            return float(eval(expr_resolved))
        except:
            return 0.0

    try:
        return float(val)
    except:
        return 0.0


def parse_range_avg(val):
    """
    Convert a value that may be a plain number, a range string ('80-100',
    '3-4'), or a datetime (Excel-misread range) into a float.
    Range strings → average of all extracted numbers.
    """
    if val is None:
        return 0.0
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, datetime.datetime):
        # Recover the original range (e.g. '3-4') from day/month
        return (val.day + val.month) / 2.0
    if not isinstance(val, str):
        return 0.0

    val = val.strip()
    if val.upper() in ('NA', 'N/A', 'NIL', 'NULL', ''):
        return 0.0

    try:
        return float(val)
    except:
        pass

    nums = re.findall(r'\d+\.?\d*', val)
    if nums:
        return sum(float(n) for n in nums) / len(nums)
    return 0.0


def get_raw_str(ws, row, col):
    """
    Return the raw string content of a cell (no formula evaluation).
    NA / NIL / None → '0'.  datetime → 'day-month' string.
    """
    val = ws.cell(row, col).value
    if val is None:
        return '0'
    if isinstance(val, (int, float)):
        return str(int(val)) if float(val) == int(float(val)) else str(val)
    if isinstance(val, datetime.datetime):
        return f'{val.day}-{val.month}'
    if isinstance(val, str):
        if val.strip().upper() in ('NA', 'N/A', 'NIL', 'NULL'):
            return '0'
        return val.strip()
    return str(val)


def multiply_comma_vals(val_str, multiplier):
    """
    Take a comma-separated string of numbers (e.g. '100, 180, 4'),
    multiply each by `multiplier`, return comma-separated result string.
    Non-numeric parts (like 'Potash' with no quantity) → 0.
    """
    if not val_str or str(val_str).strip() in ('0', ''):
        return '0'
    parts = [p.strip() for p in str(val_str).split(',')]
    results = []
    for p in parts:
        try:
            num = float(p)
            product = round(num * multiplier, 4)
            results.append(str(int(product)) if product == int(product) else str(product))
        except:
            results.append('0')
    return ', '.join(results)


def fmt(val):
    """Clean numeric formatting for cell output."""
    if isinstance(val, float) and val == int(val):
        return int(val)
    return val




# ============================================================
# CREATE Energy_Params SHEET
# ============================================================

In [ ]:
PARAM_LABELS = [
    'name',
    'survey_id',
    'crop',
    'area_ha',
    'irrigation_water_m3',
    'pump_rating',
    'pump_hrs',
    'seed_kg',
    'tractor_hrs',
    'fuel_consumption_ltr/hrs',
    'total_fuel_consumption',
    'total_man_days',
    'fert_names',
    'fert_amount_kg',
    'herbicides_ml',
    'insecticides_ml',
    'fungicides_ml',
    'yields_kg',
]

# Drop & recreate the sheet for idempotency
if 'Energy_Params' in wb.sheetnames:
    del wb['Energy_Params']
ep = wb.create_sheet('Energy_Params')

# Column A — parameter labels
for i, label in enumerate(PARAM_LABELS, start=1):
    ep.cell(i, 1, label)

# Columns B onwards — one column per farmer-crop
for col in range(2, MAX_COL + 1):
    area_ha = get_numeric(ws, 6, col)

    # 1  name
    ep.cell(1, col, ws.cell(3, col).value or 0)

    # 2  survey_id
    ep.cell(2, col, ws.cell(2, col).value or 0)

    # 3  crop
    ep.cell(3, col, ws.cell(7, col).value or 0)

    # 4  area_ha
    ep.cell(4, col, fmt(area_ha))

    # 5  irrigation_water_m3  (row 17, formula-evaluated)
    ep.cell(5, col, fmt(round(get_numeric(ws, 17, col), 2)))

    # 6  pump_rating  (row 14; NIL → 0)
    pump_raw = ws.cell(14, col).value
    pump_out = 0 if (pump_raw is None or
                     str(pump_raw).strip().upper() in ('NA', 'N/A', 'NIL', 'NULL')) \
               else fmt(get_numeric(ws, 14, col))
    ep.cell(6, col, pump_out)

    # 7  pump_hrs  (row 16, formula-evaluated)
    ep.cell(7, col, fmt(round(get_numeric(ws, 16, col), 2)))

    # 8  seed_kg  (row 18)
    ep.cell(8, col, fmt(round(get_numeric(ws, 18, col) * area_ha, 4)))

    # 9  tractor_hrs  (row 24, formula-evaluated)
    ep.cell(9, col, fmt(round(get_numeric(ws, 24, col), 4)))

    # 10 fuel_consumption_ltr/hrs  — always 0
    ep.cell(10, col, 0)

    # 11 total_fuel_consumption  (row 27, formula-evaluated)
    ep.cell(11, col, fmt(round(get_numeric(ws, 27, col), 2)))

    # 12 total_man_days = row28 × row29  (both may be ranges → averaged)
    labours  = parse_range_avg(ws.cell(28, col).value)
    man_days = parse_range_avg(ws.cell(29, col).value)
    ep.cell(12, col, fmt(round(labours * man_days, 2)))

    # 13 fert_names  (row 32, raw string)
    fert_names_raw = ws.cell(32, col).value
    if fert_names_raw is None or str(fert_names_raw).strip().upper() in ('NA', 'N/A', 'NIL', 'NULL'):
        ep.cell(13, col, 0)
    else:
        ep.cell(13, col, str(fert_names_raw).strip())

    # 14 fert_amount_kg = row33 × area_ha  (per fertilizer, comma-separated)
    ep.cell(14, col, multiply_comma_vals(get_raw_str(ws, 33, col), area_ha))

    # 15 herbicides_ml  = row36 × 1000 × area_ha
    ep.cell(15, col, multiply_comma_vals(get_raw_str(ws, 36, col), 1000 * area_ha))

    # 16 insecticides_ml = row39 × 1000 × area_ha
    ep.cell(16, col, multiply_comma_vals(get_raw_str(ws, 39, col), 1000 * area_ha))

    # 17 fungicides_ml  = row42 × 1000 × area_ha
    ep.cell(17, col, multiply_comma_vals(get_raw_str(ws, 42, col), 1000 * area_ha))

    # 18 yields_kg  (row 47 — numeric, formula, or text)
    yval = ws.cell(47, col).value
    if isinstance(yval, (int, float)):
        ep.cell(18, col, fmt(round(yval * area_ha, 4)))
    elif isinstance(yval, str) and yval.strip().startswith('='):
        ep.cell(18, col, fmt(round(get_numeric(ws, 47, col) * area_ha, 4)))
    elif yval is None:
        ep.cell(18, col, 0)
    else:
        ep.cell(18, col, str(yval))  # e.g. 'used for cattle feeding' — no multiplication

wb.save(FILE_PATH)
print(f"\n✅  Done!  'Energy_Params' sheet written to:\n   {FILE_PATH}")


✅  Done!  'Energy_Params' sheet written to:
   /content/drive/My Drive/MTP/Dataset/DIP_Data.xlsx


# For correcting other's data files in the folder

In [ ]:
# ============================================================
# Fix "total_man_days" row in Energy_Params sheet
# Run in Google Colab
# ============================================================

# from google.colab import drive
# drive.mount('/content/drive')

# import openpyxl
# import re
# import datetime

FILE_PATH = '/content/drive/My Drive/MTP/Dataset/VP_Data.xlsx'

wb = openpyxl.load_workbook(FILE_PATH)
ws_use    = wb['Energy_Use']
ws_params = wb['Energy_Params']

# ============================================================
# UTILITY
# ============================================================

def parse_range_avg(val):
    """
    Convert a value to float, handling:
      - plain number / int / float        → as-is
      - datetime (Excel misread "3-4")    → (day + month) / 2
      - range string ("80-100", "15-20")  → average of extracted numbers
      - NA / NIL / None                   → 0.0
    """
    if val is None:
        return 0.0
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, datetime.datetime):
        return (val.day + val.month) / 2.0
    if not isinstance(val, str):
        return 0.0
    val = val.strip()
    if val.upper() in ('NA', 'N/A', 'NIL', 'NULL', ''):
        return 0.0
    try:
        return float(val)
    except:
        pass
    nums = re.findall(r'\d+\.?\d*', val)
    if nums:
        return sum(float(n) for n in nums) / len(nums)
    return 0.0

def fmt(val):
    if isinstance(val, float) and val == int(val):
        return int(val)
    return val

# ============================================================
# FIND total_man_days ROW IN Energy_Params
# ============================================================

target_row = None
for row in ws_params.iter_rows():
    if row[0].value == 'total_man_days':
        target_row = row[0].row
        break

if target_row is None:
    print("❌  Could not find 'total_man_days' row in Energy_Params sheet.")
else:
    print(f"Found 'total_man_days' at row {target_row} in Energy_Params.")

    MAX_COL = ws_use.max_column
    updated = 0

    # Energy_Params col B onwards matches Energy_Use col 2 onwards (1-to-1)
    for col in range(2, MAX_COL + 1):
        labours  = parse_range_avg(ws_use.cell(32, col).value)
        man_days = parse_range_avg(ws_use.cell(33, col).value)
        new_val  = fmt(round(labours * man_days, 2))
        ws_params.cell(target_row, col, new_val)
        updated += 1

    wb.save(FILE_PATH)
    print(f"✅  Updated {updated} cells in 'total_man_days' row.")
    print(f"   Saved to: {FILE_PATH}")

Found 'total_man_days' at row 12 in Energy_Params.
✅  Updated 165 cells in 'total_man_days' row.
   Saved to: /content/drive/My Drive/MTP/Dataset/VP_Data.xlsx


# Different Crops Growmn by Farmers

In [ ]:
import openpyxl
import os

# Define the directory containing the Excel files
FOLDER_PATH = '/content/drive/My Drive/MTP/Dataset/'

# Get a list of all .xlsx files in the folder
excel_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith('.xlsx')]

# Dictionary to store aggregated crop counts across all files (case-insensitive)
all_unique_crops_counts = {}

if not excel_files:
    print(f"No Excel files found in {FOLDER_PATH}")
else:
    print(f"Found {len(excel_files)} Excel files in {FOLDER_PATH}")
    for filename in excel_files:
        current_file_path = os.path.join(FOLDER_PATH, filename)
        print(f"\n--- Processing file: {filename} ---")

        try:
            wb = openpyxl.load_workbook(current_file_path)
            if 'Energy_Params' not in wb.sheetnames:
                print(" 'Energy_Params' sheet not found in this file. Skipping.")
                continue

            ep_sheet = wb['Energy_Params']

            crop_row_found = False
            current_file_crop_counts = {}

            # Iterate through rows to find the 'crop' row
            for row_idx in range(1, ep_sheet.max_row + 1):
                if ep_sheet.cell(row=row_idx, column=1).value == 'crop':
                    crop_row_found = True
                    # Iterate through columns in the 'crop' row to get crop names and counts
                    for col_idx in range(2, ep_sheet.max_column + 1):
                        crop_name = ep_sheet.cell(row=row_idx, column=col_idx).value
                        if crop_name:
                            # Convert to lowercase and strip whitespace for case-insensitive counting
                            cleaned_crop_name = str(crop_name).strip().lower()
                            current_file_crop_counts[cleaned_crop_name] = current_file_crop_counts.get(cleaned_crop_name, 0) + 1
                            # Also update the global aggregated count
                            all_unique_crops_counts[cleaned_crop_name] = all_unique_crops_counts.get(cleaned_crop_name, 0) + 1
                    break

            if crop_row_found:
                print("Unique crops and their counts in this file (case-insensitive):")
                for crop, count in sorted(current_file_crop_counts.items()):
                    print(f"- {crop}: {count}")
            else:
                print("The 'crop' row was not found in the 'Energy_Params' sheet.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Print aggregated unique crops and their total counts across all files
print("\n========================================")
print("Aggregated Unique Crops and Total Counts Across All Files (case-insensitive):")
print("========================================")
if all_unique_crops_counts:
    for crop, count in sorted(all_unique_crops_counts.items()):
        print(f"- {crop}: {count}")
else:
    print("No crops found in any of the files.")

Found 5 Excel files in /content/drive/My Drive/MTP/Dataset/

--- Processing file: APS_Data.xlsx ---
Unique crops and their counts in this file (case-insensitive):
- arhar: 7
- ash gourd: 2
- bajra: 45
- barley: 3
- carrot: 5
- cauliflower: 1
- chilli: 1
- gram: 15
- jowar: 21
- ladyfinger: 1
- maize: 2
- mustard: 36
- peanut: 3
- peas: 2
- potato: 2
- pulses: 1
- rice: 57
- seasame: 19
- sugarcane: 5
- urad: 5
- wheat: 112

--- Processing file: MA_Data.xlsx ---
Unique crops and their counts in this file (case-insensitive):
- chilli: 22
- corn: 34
- cotton: 28
- pulses: 26
- rice: 140
- sesame seeds: 29

--- Processing file: Toko_Data.xlsx ---
Unique crops and their counts in this file (case-insensitive):
- cardamom: 15
- ginger: 5
- guava: 1
- maize: 21
- mango: 3
- miaze: 1
- milet: 1
- millet: 5
- orange: 7
- pumpkin: 7
- rice: 39
- turmeric: 1
- wheat: 1

--- Processing file: DIP_Data.xlsx ---
Unique crops and their counts in this file (case-insensitive):
- arhar: 3
- bajra: 30
- bi

In [ ]:
import openpyxl
import os

# Define the directory containing the Excel files
FOLDER_PATH = '/content/drive/My Drive/MTP/Dataset/'

# Get a list of all .xlsx files in the folder
excel_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith('.xlsx')]

# Dictionary to store aggregated fertilizer counts across all files (case-insensitive)
all_unique_fert_counts = {}

if not excel_files:
    print(f"No Excel files found in {FOLDER_PATH}")
else:
    print(f"Found {len(excel_files)} Excel files in {FOLDER_PATH}")
    for filename in excel_files:
        current_file_path = os.path.join(FOLDER_PATH, filename)
        print(f"\n--- Processing file: {filename} ---")

        try:
            wb = openpyxl.load_workbook(current_file_path)
            if 'Energy_Params' not in wb.sheetnames:
                print(" 'Energy_Params' sheet not found in this file. Skipping.")
                continue

            ep_sheet = wb['Energy_Params']

            fert_row_found = False
            current_file_fert_counts = {}

            # Iterate through rows to find the 'fert_names' row
            for row_idx in range(1, ep_sheet.max_row + 1):
                if ep_sheet.cell(row=row_idx, column=1).value == 'fert_names':
                    fert_row_found = True
                    # Iterate through columns in the 'fert_names' row
                    for col_idx in range(2, ep_sheet.max_column + 1):
                        fert_names_str = ep_sheet.cell(row=row_idx, column=col_idx).value
                        if fert_names_str and str(fert_names_str).strip().upper() not in ('0', 'NA', 'N/A', 'NIL', 'NULL', ''):
                            # Replace '/' with ',' to treat them as common separators
                            cleaned_fert_names_str = str(fert_names_str).replace('/', ',')
                            # Split by comma and clean each individual fertilizer name
                            individual_ferts = [f.strip().lower() for f in cleaned_fert_names_str.split(',') if f.strip()]
                            for fert in individual_ferts:
                                current_file_fert_counts[fert] = current_file_fert_counts.get(fert, 0) + 1
                                # Also update the global aggregated count
                                all_unique_fert_counts[fert] = all_unique_fert_counts.get(fert, 0) + 1
                    break

            if fert_row_found:
                print("Unique fertilizers and their counts in this file (case-insensitive):")
                for fert, count in sorted(current_file_fert_counts.items()):
                    print(f"- {fert}: {count}")
            else:
                print("The 'fert_names' row was not found in the 'Energy_Params' sheet.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Print aggregated unique fertilizers and their total counts across all files
print("\n========================================")
print("Aggregated Unique Fertilizers and Total Counts Across All Files (case-insensitive):")
print("========================================")
if all_unique_fert_counts:
    for fert, count in sorted(all_unique_fert_counts.items()):
        print(f"- {fert}: {count}")
else:
    print("No fertilizers found in any of the files.")


Found 5 Excel files in /content/drive/My Drive/MTP/Dataset/

--- Processing file: APS_Data.xlsx ---
Unique fertilizers and their counts in this file (case-insensitive):
- dap: 341
- organic fertilizer: 1
- potash: 69
- sulphur: 205
- urea: 340
- zinc: 224

--- Processing file: MA_Data.xlsx ---
Unique fertilizers and their counts in this file (case-insensitive):
- dap: 279
- urea: 279

--- Processing file: Toko_Data.xlsx ---
Unique fertilizers and their counts in this file (case-insensitive):

--- Processing file: VP_Data.xlsx ---
Unique fertilizers and their counts in this file (case-insensitive):
- boron: 1
- dap: 151
- mop: 44
- mpp(0-52-34): 1
- no: 3
- potash: 1
- ssp: 45
- sulphur: 12
- sulpur: 3
- suphur: 1
- urea: 99
- zinc: 3

--- Processing file: DIP_Data.xlsx ---
Unique fertilizers and their counts in this file (case-insensitive):
- ammonium sulphate: 1
- biozinc: 3
- calcium: 1
- dap: 305
- dpa: 7
- dum3: 1
- iron: 2
- jaivik khad: 3
- manure: 206
- monozinc: 5
- npk: 11
- p

In [ ]:
import openpyxl
import os

# --- IMPORTANT: Update this to the path of the specific Excel file you want to modify ---
FILE_PATH = '/content/drive/My Drive/MTP/Dataset/Toko_Data.xlsx' # Example: Change to your target file
# --------------------------------------------------------------------------------------

try:
    wb = openpyxl.load_workbook(FILE_PATH)

    if 'Energy_Use' not in wb.sheetnames:
        print(f"❌ Error: 'Energy_Use' sheet not found in {FILE_PATH}. Skipping this file.")
        exit()
    if 'Energy_Params' not in wb.sheetnames:
        print(f"❌ Error: 'Energy_Params' sheet not found in {FILE_PATH}. Skipping this file.")
        exit()

    ws_use = wb['Energy_Use']
    ws_params = wb['Energy_Params']

    # Find the row index for 'engine_type' in 'Energy_Params'
    engine_type_row_idx = None
    for r_idx in range(1, ws_params.max_row + 1):
        if str(ws_params.cell(row=r_idx, column=1).value).strip().lower() == 'engine_type':
            engine_type_row_idx = r_idx
            break

    # If 'engine_type' row not found, add it at the end
    if engine_type_row_idx is None:
        engine_type_row_idx = ws_params.max_row + 1
        ws_params.cell(row=engine_type_row_idx, column=1, value='engine_type')
        print(f"'engine_type' row added at position {engine_type_row_idx} in 'Energy_Params' sheet.")
    else:
        print(f"'engine_type' row found at position {engine_type_row_idx} in 'Energy_Params' sheet. Updating values.")

    # Determine the maximum column in Energy_Use to iterate through
    max_col_use = ws_use.max_column
    updated_cells_count = 0

    # Iterate from the second column (B) onwards to copy values
    # Energy_Params col B onwards corresponds to Energy_Use col 2 onwards
    for col_idx in range(2, max_col_use + 1):
        # --- Option 1: Directly write "electric pump" ---
        # Uncomment the line below to set all 'engine_type' values to 'electric pump'
        engine_type_value = "electric pump"

        # --- Option 2: Copy from 'Energy_Use' sheet (original behavior) ---
        # Uncomment the line below to copy values from row 15 of 'Energy_Use'
        # engine_type_value = ws_use.cell(row=15, column=col_idx).value

        # Copy the value to 'Energy_Params' sheet at the identified row
        ws_params.cell(row=engine_type_row_idx, column=col_idx, value=engine_type_value)
        updated_cells_count += 1

    wb.save(FILE_PATH)
    print(f"\n✅ Successfully updated {updated_cells_count} 'engine_type' values in 'Energy_Params' sheet of: {FILE_PATH}")

except FileNotFoundError:
    print(f"❌ Error: The file {FILE_PATH} was not found. Please check the path.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

'engine_type' row added at position 19 in 'Energy_Params' sheet.

✅ Successfully updated 127 'engine_type' values in 'Energy_Params' sheet of: /content/drive/My Drive/MTP/Dataset/Toko_Data.xlsx
